In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, TaskType, get_peft_model

In [ ]:
model_id = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_id)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
!pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 65.4 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
peft_config = LoraConfig(
    r=1,
    lora_alpha=32,
    lora_dropout=0.0,
    # task_type=TaskType.CAUSAL_LM,

    # Let PEFT automatically select the linear layers.
    # target_modules="all-linear",
    layers_to_transform=None
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()


trainable params: 136,192 || all params: 1,543,850,496 || trainable%: 0.0088


In [ ]:
dataset = load_dataset("openai/gsm8k", "main")

train_dataset = dataset["train"]
test_dataset = dataset["test"]

README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 2.31MB            

main/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

main/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  419kB            

main/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
def format_example(example):
    """
    Qwen chat format:

    user:
        problem

    assistant:
        gold chain-of-thought + final answer
    """

    messages = [
        {
            "role": "user",
            "content": example["question"],
        },
        {
            "role": "assistant",
            "content": example["answer"],
        },
    ]

    # Full conversation including the gold answer
    full_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )

    # Prompt only, ending immediately before assistant response
    prompt_text = tokenizer.apply_chat_template(
        [messages[0]],
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt_text, full_text

In [ ]:
MAX_LENGTH = 1024


def tokenize_example(example):

    prompt_text, full_text = format_example(example)

    prompt_tokens = tokenizer(
        prompt_text,
        add_special_tokens=False,
    )

    full_tokens = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LENGTH,
    )

    input_ids = full_tokens["input_ids"]
    attention_mask = full_tokens["attention_mask"]

    # Loss only on assistant/gold-answer tokens.
    prompt_len = len(prompt_tokens["input_ids"])

    labels = [-100] * prompt_len + input_ids[prompt_len:]

    # In case truncation caused an edge case
    labels = labels[:len(input_ids)]

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }


train_tokenized = train_dataset.map(
    tokenize_example,
    remove_columns=train_dataset.column_names,
)

test_tokenized = test_dataset.map(
    tokenize_example,
    remove_columns=test_dataset.column_names,
)


Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
)

In [ ]:
training_args = TrainingArguments(
    output_dir="./qwen2.5-1.5b-gsm8k-r1",

    # One epoch is a reasonable starting point
    num_train_epochs=1,

    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    learning_rate=1e-4,
    weight_decay=0.0,

    bf16=True,

    logging_steps=25,
    save_strategy="epoch",

    report_to="none",

    # Important for reproducibility
    seed=42,
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    data_collator=data_collator,
)


trainer.train()

Step,Training Loss
25,0.526741
50,0.394197
75,0.377218
100,0.370189
125,0.366510
150,0.367401
175,0.354139
200,0.361109
225,0.358113
250,0.360988


TrainOutput(global_step=468, training_loss=0.36759474338629305, metrics={'train_runtime': 122.1411, 'train_samples_per_second': 61.183, 'train_steps_per_second': 3.832, 'total_flos': 1.68964937102592e+16, 'train_loss': 0.36759474338629305, 'epoch': 1.0})

In [ ]:
model.save_pretrained("./qwen2.5-1.5b-gsm8k-r1")
tokenizer.save_pretrained("./qwen2.5-1.5b-gsm8k-r1")

('./qwen2.5-1.5b-gsm8k-r1/tokenizer_config.json',
 './qwen2.5-1.5b-gsm8k-r1/chat_template.jinja',
 './qwen2.5-1.5b-gsm8k-r1/tokenizer.json')

Eval

In [ ]:
ds = test_dataset
print(ds)

Dataset({
    features: ['question', 'answer'],
    num_rows: 1319
})


In [ ]:
import re, torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM

eval_ds = ds.select(range(150))

def extract_answer(text):
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")
    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else None

def evaluate(model, name):
    model.eval()
    correct, lengths = 0, []

    for ex in tqdm(eval_ds, desc=f"Evaluating {name}"):
        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content":
              f"Solve the following math problem. Show your reasoning and give the final answer.\n\n{ex['question']}"}],
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to(model.device)

        prompt_len = inputs["input_ids"].shape[-1]

        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=512,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(
            out[0, prompt_len:],
            skip_special_tokens=True
        )

        pred = extract_answer(response)
        gold = extract_answer(ex["answer"])

        correct += pred == gold
        lengths.append(
            len(tokenizer(response, add_special_tokens=False)["input_ids"])
        )

    print(
        f"{name:6s} | accuracy={correct/len(eval_ds):.3f} "
        f"| mean tokens={sum(lengths)/len(lengths):.1f}"
    )

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

evaluate(base_model, "Base")
evaluate(model, "SFT")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Evaluating Base:   0%|          | 0/150 [00:00<?, ?it/s]

Base   | accuracy=0.620 | mean tokens=307.7


Evaluating SFT:   0%|          | 0/150 [00:00<?, ?it/s]

SFT    | accuracy=0.480 | mean tokens=155.3


GRPO

In [ ]:
from datasets import Dataset

grpo_ds = dataset.remove_columns([]).map(
    lambda x: {
        "prompt": [
            {
                "role": "user",
                "content": (
                    "Solve the following math problem. "
                    "Show your reasoning and give the final answer.\n\n"
                    + x["question"]
                )
            }
        ],
        "ground_truth": x["answer"],
    }
)

# Use training data, NOT your 150-example evaluation set
# Ideally this should be the GSM8K train split.
print(grpo_ds)

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['question', 'answer', 'prompt', 'ground_truth'],
        num_rows: 7473
    })
    test: Dataset({
        features: ['question', 'answer', 'prompt', 'ground_truth'],
        num_rows: 1319
    })
})


In [ ]:
train_grpo_ds = grpo_ds["train"]
test_grpo_ds = grpo_ds["test"]

In [ ]:
import re

def extract_answer(text):
    m = re.search(r"####\s*([-+]?\d[\d,]*(?:\.\d+)?)", text)
    if m:
        return m.group(1).replace(",", "")

    nums = re.findall(r"[-+]?\d[\d,]*(?:\.\d+)?", text)
    return nums[-1].replace(",", "") if nums else None


def gsm8k_reward(completions, ground_truth, **kwargs):
    rewards = []

    for completion, gold in zip(completions, ground_truth):
        # TRL may provide conversational completions
        if isinstance(completion, list):
            text = completion[-1]["content"]
        else:
            text = completion

        pred = extract_answer(text)
        gold = extract_answer(gold)

        rewards.append(1.0 if pred == gold else 0.0)

    return rewards

In [ ]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 31.8 MB/s eta 0:00:00


In [ ]:
from trl import GRPOTrainer, GRPOConfig

grpo_config = GRPOConfig(
    output_dir="./qwen-grpo-gsm8k",

    learning_rate=1e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    num_generations=4,
    max_completion_length=512,

    num_train_epochs=1,

    logging_steps=10,
    save_strategy="steps",
    save_steps=100,

    bf16=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    reward_funcs=gsm8k_reward,
    args=grpo_config,
    train_dataset=train_grpo_ds,
)

trainer.train()

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,0.049242
20,0.007221
30,-0.000116
40,0.046165
50,0.094774
60,0.014078
70,-0.000491
80,0.000000
90,-0.012331
100,-0.005758


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RuntimeError: [enforce fail at inline_container.cc:672] . unexpected pos 7317097152 vs 7317097048

In [ ]:
grpo_model = trainer.model

In [ ]:
evaluate(grpo_model, "GRPO")